In [1]:
import sys, json, logging
from pathlib import Path
import yaml
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/home/yli94/CLIF/OHCA-RL")
CODE_DIR     = PROJECT_ROOT / "code"
CONFIG_DIR   = PROJECT_ROOT / "config"
OUT_DIR      = PROJECT_ROOT / "output" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Site config
with open(CONFIG_DIR / "config.json") as f:
    site_config = json.load(f)
SITE_NAME, TABLES_PATH, FILE_TYPE, TIMEZONE = (
    site_config["site_name"], site_config["tables_path"],
    site_config["file_type"], site_config["timezone"]
)

# Variable config
def deep_merge(base, override):
    out = dict(base) if base else {}
    for k, v in (override or {}).items():
        out[k] = deep_merge(out[k], v) if (k in out and isinstance(out[k], dict)
                                            and isinstance(v, dict)) else v
    return out

with open(CONFIG_DIR / "ohca_rl_config.yaml") as f:
    _base = yaml.safe_load(f)
_local_path = CONFIG_DIR / "ohca_rl_config_local.yaml"
_local = (yaml.safe_load(open(_local_path)) or {}) if _local_path.exists() else {}
ohca_config = deep_merge(_base, _local)

# utils
sys.path.insert(0, str(CODE_DIR))
for _stale in ("utils",):
    if _stale in sys.modules:
        del sys.modules[_stale]
import utils

# clifpy SOFA
import clifpy
from clifpy.utils.sofa import (
    REQUIRED_SOFA_CATEGORIES_BY_TABLE, compute_sofa
)
from clifpy import (Labs, Vitals, MedicationAdminContinuous,
                    RespiratorySupport, PatientAssessments)
from clifpy.utils.outlier_handler import apply_outlier_handling

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("02_sofa")

# Load cohort + anchor mapping
cohort_df  = pd.read_parquet(OUT_DIR / "cohort_ohca_icu.parquet")
anchor_map = pd.read_parquet(OUT_DIR / "anchor_mapping.parquet")
cohort_hosp_ids = cohort_df["hospitalization_id"].astype(str).unique().tolist()

print(f"Cohort        : {len(cohort_hosp_ids):,} hospitalizations")
print(f"Anchor map    : {len(anchor_map):,} rows (one per patient)")
print(f"Anchor coverage: {anchor_map['anchor_dttm'].notna().sum():,}/{len(anchor_map):,}")
print(f"clifpy        : {clifpy.__version__}")

Cohort        : 1,457 hospitalizations
Anchor map    : 1,457 rows (one per patient)
Anchor coverage: 1,456/1,457
clifpy        : 0.3.9


In [2]:
# clifpy's SOFA needs labs, vitals, meds_cont, resp_support, patient_assessments.
# Pull them with SOFA-required categories (clifpy provides the exact list).
sofa_cats = REQUIRED_SOFA_CATEGORIES_BY_TABLE
print("SOFA-required categories per table:")
for tbl, cats in sofa_cats.items():
    print(f"  {tbl}: {cats}")

SOFA-required categories per table:
  labs: ['creatinine', 'platelet_count', 'po2_arterial', 'bilirubin_total']
  vitals: ['map', 'spo2']
  patient_assessments: ['gcs_total']
  medication_admin_continuous: ['norepinephrine', 'epinephrine', 'dopamine', 'dobutamine']
  respiratory_support: ['device_category', 'fio2_set']


In [3]:
# Pull labs (union of OHCA labs + SOFA-required labs)
_sofa_labs = sorted(set(ohca_config["labs_of_interest"]) | set(sofa_cats["labs"]))
print(f"Loading labs for SOFA + OHCA ({len(_sofa_labs)} cats)...")

labs_tbl = Labs.from_file(
    data_directory=TABLES_PATH, filetype=FILE_TYPE, timezone=TIMEZONE,
    filters={"hospitalization_id": cohort_hosp_ids,
             "lab_category": _sofa_labs},
)
apply_outlier_handling(labs_tbl)

vitals_tbl = Vitals.from_file(
    data_directory=TABLES_PATH, filetype=FILE_TYPE, timezone=TIMEZONE,
    filters={"hospitalization_id": cohort_hosp_ids,
             "vital_category": sofa_cats["vitals"]},
)
apply_outlier_handling(vitals_tbl)

meds_tbl = MedicationAdminContinuous.from_file(
    data_directory=TABLES_PATH, filetype=FILE_TYPE, timezone=TIMEZONE,
    filters={"hospitalization_id": cohort_hosp_ids,
             "med_category": sofa_cats["medication_admin_continuous"]},
)
apply_outlier_handling(meds_tbl)

resp_tbl = RespiratorySupport.from_file(
    data_directory=TABLES_PATH, filetype=FILE_TYPE, timezone=TIMEZONE,
    filters={"hospitalization_id": cohort_hosp_ids},
)
apply_outlier_handling(resp_tbl)

assess_tbl = PatientAssessments.from_file(
    data_directory=TABLES_PATH, filetype=FILE_TYPE, timezone=TIMEZONE,
    filters={"hospitalization_id": cohort_hosp_ids,
             "assessment_category": sofa_cats["patient_assessments"]},
)
apply_outlier_handling(assess_tbl)

# Coverage check for the SOFA-required labs
print(f"\nSOFA lab coverage in MIMIC cohort:")
_lab_df = labs_tbl.df
_lab_df["lab_category"] = _lab_df["lab_category"].str.lower()
for _c in sofa_cats["labs"]:
    _n = (_lab_df["lab_category"] == _c).sum()
    _pts = _lab_df.loc[_lab_df["lab_category"] == _c, "hospitalization_id"].astype(str).nunique()
    print(f"  {_c:20s} {_n:>8,} rows  {_pts:>5,} patients")

Loading labs for SOFA + OHCA (17 cats)...
📢 Initialized labs table
📢 Data directory: /home/yli94/CLIF_MIMIC
📢 File type: parquet
📢 Timezone: US/Eastern
📢 Output directory: /home/yli94/CLIF/OHCA-RL/notebook/output
📢 Loaded schema from /home/yli94/CLIF/OHCA-RL/.venv/lib/python3.13/site-packages/clifpy/schemas/labs_schema.yaml
📢 Loaded outlier configuration
Using CLIF standard outlier ranges

Building outlier expressions...


Building expressions: 100%|██████████| 1/1 [00:00<00:00, 162.95column/s]


Applying outlier filtering...


Processing: 100%|██████████| 1/1 [00:00<00:00,  8.44operation/s]



Labs Table - Category Statistics:
  bicarbonate         :  32216 values →      0 nullified (  0.0%)
  bilirubin_total     :  11469 values →      0 nullified (  0.0%)
  bun                 :  31974 values →      4 nullified (  0.0%)
  calcium_total       :  30121 values →      4 nullified (  0.0%)
  chloride            :  33895 values →      1 nullified (  0.0%)
  creatinine          :  32533 values →      1 nullified (  0.0%)
  glucose_serum       :  40678 values →      3 nullified (  0.0%)
  hemoglobin          :  32253 values →      2 nullified (  0.0%)
  lactate             :  19659 values →      0 nullified (  0.0%)
  magnesium           :  31639 values →      6 nullified (  0.0%)
  pco2_arterial       :  27829 values →      1 nullified (  0.0%)
  ph_arterial         :  28838 values →      3 nullified (  0.0%)
  platelet_count      :  28706 values →      0 nullified (  0.0%)
  po2_arterial        :  27823 values →      1 nullified (  0.0%)
  potassium           :  41596 values →  

...

Building expressions: 100%|██████████| 1/1 [00:00<00:00, 809.71column/s]


Applying outlier filtering...


Processing: 100%|██████████| 1/1 [00:00<00:00, 55.03operation/s]


Patient Assessments Table - Category Statistics:
  gcs_total           :  55920 values →      0 nullified (  0.0%)

SOFA lab coverage in MIMIC cohort:


  creatinine             32,541 rows  1,422 patients
  platelet_count         28,821 rows  1,417 patients
  po2_arterial           27,833 rows  1,345 patients
  bilirubin_total        11,469 rows  1,312 patients


In [4]:
# SOFA's cardiovascular subscore checks vasopressor doses in mcg/kg/min.
# Use the same dose conversion pipeline as notebook 01.
weight_df = pd.read_parquet(OUT_DIR / "weight_lookup.parquet")

_sofa_meds_preferred = {
    "norepinephrine": "mcg/kg/min",
    "epinephrine":    "mcg/kg/min",
    "dopamine":       "mcg/kg/min",
    "dobutamine":     "mcg/kg/min",
}

meds_df = meds_tbl.df.copy()
meds_df["hospitalization_id"] = meds_df["hospitalization_id"].astype(str)
meds_df["med_category"]       = meds_df["med_category"].str.lower()
meds_df = meds_df[meds_df["med_dose"].notna() & (meds_df["med_dose"] >= 0)].copy()

print("Pre-conversion unit diversity:")
print(meds_df.groupby(["med_category", "med_dose_unit"]).size()
      .reset_index(name="n").to_string(index=False))

meds_df, _conv = utils.convert_med_doses(meds_df, weight_df, _sofa_meds_preferred)
if "med_dose_converted" in meds_df.columns:
    meds_df["med_dose"] = meds_df["med_dose_converted"]

# Re-attach to the clifpy table so SOFA gets the converted doses
meds_tbl.df = meds_df

print(f"\nPost-conversion: {len(meds_df):,} rows")
print(meds_df.groupby("med_category")["med_dose"].describe(percentiles=[.5, .9])
      .round(3).to_string())

Pre-conversion unit diversity:
  med_category med_dose_unit     n
    dobutamine    mcg/kg/min  1191
      dopamine    mcg/kg/min   736
   epinephrine    mcg/kg/min  5376
norepinephrine    mcg/kg/min 35370


2026-05-30 02:35:28,976 | WARNING | Imputed weight_kg with cohort median (90.3 kg) for 3 rows across 1 patients with NO weight ever recorded
2026-05-30 02:35:29,420 | INFO | Dose conversion complete: 42673/42673 rows successful (100.0%)
2026-05-30 02:35:29,422 | INFO |   dobutamine: 1191 rows converted mcg/kg/min → mcg/kg/min
2026-05-30 02:35:29,422 | INFO |   dopamine: 736 rows converted mcg/kg/min → mcg/kg/min
2026-05-30 02:35:29,423 | INFO |   epinephrine: 5376 rows converted mcg/kg/min → mcg/kg/min
2026-05-30 02:35:29,423 | INFO |   norepinephrine: 35370 rows converted mcg/kg/min → mcg/kg/min



Post-conversion: 42,673 rows
                  count   mean    std  min    50%     90%     max
med_category                                                     
dobutamine       1191.0  3.076  3.514  0.0  2.500   7.548  19.850
dopamine          736.0  6.482  6.597  0.0  5.000  19.611  27.113
epinephrine      5376.0  0.158  0.288  0.0  0.050   0.492   2.000
norepinephrine  35370.0  0.138  0.166  0.0  0.082   0.345   2.891


In [5]:
# REVIEWED: Build per-patient 0-24h window. Original notebook had a second
# compute_sofa() call here using the table-object API that doesn't exist in clifpy
# — it raised TypeError silently. We removed the dead call and now define
# `_cohort_window` (a DataFrame, as required by clifpy.compute_sofa's cohort_df arg)
# in this cell so the next cell can use it without relying on stale kernel state.

anchor_with_window = anchor_map[["hospitalization_id", "anchor_dttm"]].copy()
anchor_with_window["start_time"] = anchor_with_window["anchor_dttm"]
anchor_with_window["end_time"]   = anchor_with_window["anchor_dttm"] + pd.Timedelta(hours=24)
anchor_with_window["hospitalization_id"] = anchor_with_window["hospitalization_id"].astype(str)
anchor_with_window = anchor_with_window.dropna(subset=["anchor_dttm"]).copy()

_cohort_window = anchor_with_window[["hospitalization_id", "start_time", "end_time"]].copy()
print(f"Built _cohort_window: {len(_cohort_window):,} patients")
print(f"  Start range: {_cohort_window['start_time'].min()} → {_cohort_window['start_time'].max()}")
print(f"  Sample rows:")
print(_cohort_window.head(3).to_string(index=False))

Built _cohort_window: 1,456 patients
  Start range: 2110-02-08 00:12:00-05:00 → 2211-12-27 00:06:00-05:00
  Sample rows:
hospitalization_id                start_time                  end_time
          20186322 2160-01-17 03:24:00-05:00 2160-01-18 03:24:00-05:00
          29269604 2112-05-01 10:44:00-05:00 2112-05-02 10:44:00-05:00
          26609648 2145-03-17 16:15:00-05:00 2145-03-18 16:15:00-05:00


In [6]:
# clifpy's compute_sofa expects a single wide DataFrame with all SOFA inputs
# already pivoted. Med columns must follow the naming convention
# `{med}_mcg_kg_min` (e.g. norepinephrine_mcg_kg_min).

# ── (a) Pivot labs (already lowercased above) ──
_labs_long = labs_tbl.df.copy()
_labs_long["hospitalization_id"] = _labs_long["hospitalization_id"].astype(str)
_labs_long["lab_category"] = _labs_long["lab_category"].str.lower()
labs_for_sofa = _labs_long[_labs_long["lab_category"].isin(sofa_cats["labs"])].copy()

labs_sofa_wide = labs_for_sofa.pivot_table(
    index=["hospitalization_id", "lab_result_dttm"],
    columns="lab_category", values="lab_value_numeric", aggfunc="first",
).reset_index().rename(columns={"lab_result_dttm": "event_dttm"})

# ── (b) Pivot vitals ──
_vit_long = vitals_tbl.df.copy()
_vit_long["hospitalization_id"] = _vit_long["hospitalization_id"].astype(str)
_vit_long["vital_category"] = _vit_long["vital_category"].str.lower()

vitals_sofa_wide = _vit_long.pivot_table(
    index=["hospitalization_id", "recorded_dttm"],
    columns="vital_category", values="vital_value", aggfunc="first",
).reset_index().rename(columns={"recorded_dttm": "event_dttm"})

# ── (c) Pivot meds with the required naming convention ──
meds_sofa_wide = meds_df.pivot_table(
    index=["hospitalization_id", "admin_dttm"],
    columns="med_category", values="med_dose", aggfunc="first",
).reset_index().rename(columns={"admin_dttm": "event_dttm"})
# clifpy expects `{med}_mcg_kg_min`
meds_sofa_wide.columns = [
    f"{c}_mcg_kg_min" if c in sofa_cats["medication_admin_continuous"] else c
    for c in meds_sofa_wide.columns
]

# ── (d) Respiratory: device_category + fio2_set (post-waterfall) ──
print("Running respiratory waterfall for SOFA...")
resp_tbl_sofa = resp_tbl.waterfall()
_resp = resp_tbl_sofa.df.copy()
_resp["hospitalization_id"] = _resp["hospitalization_id"].astype(str)
resp_sofa_wide = _resp[["hospitalization_id", "recorded_dttm",
                        "device_category", "fio2_set"]].rename(
    columns={"recorded_dttm": "event_dttm"}
)

# ── (e) Assessments (gcs_total) ──
_assess = assess_tbl.df.copy()
_assess["hospitalization_id"] = _assess["hospitalization_id"].astype(str)
_assess["assessment_category"] = _assess["assessment_category"].str.lower()

assess_sofa_wide = _assess[_assess["assessment_category"] == "gcs_total"].pivot_table(
    index=["hospitalization_id", "recorded_dttm"],
    columns="assessment_category", values="numerical_value", aggfunc="first",
).reset_index().rename(columns={"recorded_dttm": "event_dttm"})

# ── (f) Outer-merge all five sources on (hospitalization_id, event_dttm) ──
sofa_wide = labs_sofa_wide.copy()
for _name, _df in [("vitals", vitals_sofa_wide), ("meds", meds_sofa_wide),
                   ("resp", resp_sofa_wide), ("assess", assess_sofa_wide)]:
    if len(_df):
        # Normalize tz
        if pd.api.types.is_datetime64_any_dtype(_df["event_dttm"]):
            if _df["event_dttm"].dt.tz is None:
                _df = _df.copy()
                _df["event_dttm"] = pd.to_datetime(_df["event_dttm"], utc=True)
        sofa_wide = sofa_wide.merge(_df, on=["hospitalization_id", "event_dttm"],
                                    how="outer")

sofa_wide = sofa_wide.sort_values(["hospitalization_id", "event_dttm"]).reset_index(drop=True)
print(f"SOFA wide df: {len(sofa_wide):,} rows × {sofa_wide.shape[1]} cols")
print(f"Columns: {sorted(sofa_wide.columns.tolist())}")

Running respiratory waterfall for SOFA...
Converting timezone from US/Eastern to UTC for waterfall processing
✦ Phase 0: initialise & create hourly scaffold
  • Building hourly scaffold via DuckDB
  • Scaffold rows created: 273,523
✦ Phase 1: heuristic inference of device & mode
6 rows had PEEP>0 on nasal cannula device_category reset
✦ Phase 2: build hierarchical IDs
✦ Phase 3: forward-only numeric fill inside mode_name_id blocks
  • applying waterfall fill to 1,380 encounters


Waterfall fill by mode_name_id:  99%|█████████▉| 10136/10235 [00:07<00:00, 981.07it/s]/home/yli94/CLIF/OHCA-RL/.venv/lib/python3.13/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
Waterfall fill by mode_name_id: 100%|██████████| 10235/10235 [00:09<00:00, 1116.88it/s]


✦ Phase 4: final dedup & ordering
[OK] Respiratory-support waterfall complete.
Converting timezone from UTC back to US/Eastern after processing
📢 Initialized respiratory_support table
📢 Data directory: /home/yli94/CLIF_MIMIC
📢 File type: parquet
📢 Timezone: US/Eastern
📢 Output directory: /home/yli94/CLIF/OHCA-RL/notebook/output
📢 Loaded schema from /home/yli94/CLIF/OHCA-RL/.venv/lib/python3.13/site-packages/clifpy/schemas/respiratory_support_schema.yaml
📢 Loaded outlier configuration
SOFA wide df: 829,665 rows × 15 cols
Columns: ['bilirubin_total', 'creatinine', 'device_category', 'dobutamine_mcg_kg_min', 'dopamine_mcg_kg_min', 'epinephrine_mcg_kg_min', 'event_dttm', 'fio2_set', 'gcs_total', 'hospitalization_id', 'map', 'norepinephrine_mcg_kg_min', 'platelet_count', 'po2_arterial', 'spo2']


In [7]:
# clifpy compute_sofa expects the timestamp column named `event_time`
sofa_wide_for_sofa = sofa_wide.rename(columns={"event_dttm": "event_time"})

print(f"Computing 0-24h SOFA for {len(_cohort_window):,} patients...")

# Harmonize tz between sofa_wide_for_sofa and the cohort window
if sofa_wide_for_sofa["event_time"].dt.tz is not None and _cohort_window["start_time"].dt.tz is None:
    _cohort_window["start_time"] = _cohort_window["start_time"].dt.tz_localize(TIMEZONE).dt.tz_convert("UTC")
    _cohort_window["end_time"]   = _cohort_window["end_time"].dt.tz_localize(TIMEZONE).dt.tz_convert("UTC")
elif sofa_wide_for_sofa["event_time"].dt.tz is None and _cohort_window["start_time"].dt.tz is not None:
    sofa_wide_for_sofa["event_time"] = sofa_wide_for_sofa["event_time"].dt.tz_localize("UTC")

sofa_0_24 = compute_sofa(
    sofa_wide_for_sofa,
    cohort_df=_cohort_window,
    id_name="hospitalization_id",
    extremal_type="worst",
    fill_na_scores_with_zero=True,
    remove_outliers=False,
)

# Tag columns with the window label
_score_cols = [c for c in sofa_0_24.columns if c != "hospitalization_id"]
sofa_0_24 = sofa_0_24.rename(columns={c: f"{c}_0_24" for c in _score_cols})

print(f"\nSOFA 0-24h computed for {len(sofa_0_24):,} patients")
print(f"\nColumns: {list(sofa_0_24.columns)}")

# Identify the total column (clifpy may name it sofa_total or similar)
_tot_cols = [c for c in sofa_0_24.columns if "total" in c.lower()]
if _tot_cols:
    print(f"\nTotal SOFA ({_tot_cols[0]}) distribution:")
    print(sofa_0_24[_tot_cols[0]].describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

print(f"\nAll subscore medians:")
for c in [c for c in sofa_0_24.columns if c != "hospitalization_id"]:
    print(f"  {c:40s} median={sofa_0_24[c].median():.1f}")

# Save
sofa_0_24.to_parquet(OUT_DIR / "sofa_0_24_reviewed.parquet", index=False)
print(f"\nSaved → {OUT_DIR / 'sofa_0_24.parquet'}")

Computing 0-24h SOFA for 1,456 patients...

SOFA 0-24h computed for 1,424 patients

Columns: ['hospitalization_id', 'p_f_0_24', 'p_f_imputed_0_24', 'sofa_cv_97_0_24', 'sofa_coag_0_24', 'sofa_liver_0_24', 'sofa_resp_0_24', 'sofa_cns_0_24', 'sofa_renal_0_24', 'sofa_total_0_24']

Total SOFA (sofa_total_0_24) distribution:
count    1424.0
mean        4.2
std         3.0
min         0.0
10%         1.0
25%         2.0
50%         4.0
75%         6.0
90%         8.0
max        14.0

All subscore medians:
  p_f_0_24                                 median=75.0
  p_f_imputed_0_24                         median=75.7
  sofa_cv_97_0_24                          median=1.0
  sofa_coag_0_24                           median=0.0
  sofa_liver_0_24                          median=0.0
  sofa_resp_0_24                           median=0.0
  sofa_cns_0_24                            median=0.0
  sofa_renal_0_24                          median=1.0
  sofa_total_0_24                          median=4.0

Saved →

In [8]:
# Pick 5 patients with the highest mortality risk (expired) and inspect SOFA
_dead_ids = cohort_df.loc[cohort_df["survival_status"] == "non-survivor",
                          "hospitalization_id"].astype(str).head(5).tolist()
print("SOFA 0-24h for 5 patients who died in-hospital:")
print(sofa_0_24[sofa_0_24["hospitalization_id"].isin(_dead_ids)].to_string(index=False))

print("\nFor reference, look at one of these patients' raw GCS values in first 24h:")
_pid = _dead_ids[0]
_anchor = anchor_map.loc[anchor_map["hospitalization_id"] == _pid, "anchor_dttm"].iloc[0]
_window_end = _anchor + pd.Timedelta(hours=24)

_assess_pt = assess_tbl.df.copy()
_assess_pt["hospitalization_id"] = _assess_pt["hospitalization_id"].astype(str)
_assess_pt = _assess_pt[
    (_assess_pt["hospitalization_id"] == _pid) &
    (_assess_pt["recorded_dttm"] >= _anchor) &
    (_assess_pt["recorded_dttm"] <= _window_end) &
    (_assess_pt["assessment_category"].str.lower() == "gcs_total")
].sort_values("recorded_dttm")
print(f"\nPatient {_pid}: GCS values in first 24h (anchor={_anchor}):")
print(_assess_pt[["recorded_dttm", "numerical_value"]].head(20).to_string(index=False))

SOFA 0-24h for 5 patients who died in-hospital:
hospitalization_id  p_f_0_24  p_f_imputed_0_24  sofa_cv_97_0_24  sofa_coag_0_24  sofa_liver_0_24  sofa_resp_0_24  sofa_cns_0_24  sofa_renal_0_24  sofa_total_0_24
          20186322      62.0        127.573449                3               1                0               0              0                2                6
          23950803       NaN               NaN                0               0                0               0              0                2                2
          21550895       NaN               NaN                0               1                0               0              0                0                1
          21676306     100.0               NaN                4               0                1               0              0                2                7

For reference, look at one of these patients' raw GCS values in first 24h:

Patient 20186322: GCS values in first 24h (anchor=2160-01-17 03:2